# Lecture 4 (26 Sep 2026) — Part 1: NumPy

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kosavina/hse-python-2026/blob/main/Lecture_4.ipynb)

From now on the model writes most of the NumPy. Your job is to **read it**: when a NumPy line looks right and is wrong, the reason is almost always one of three things.

- **shape** — how many rows and columns came out? (`.shape`)
- **dtype** — are these floats, integers, or strings? (`.dtype`)
- **axis** — did it collapse the rows or the columns? (`axis=0` / `axis=1`)

By the end of Part 1 you should be able to look at any NumPy line and say what shape comes out. Part 2 puts that to work on real problems with the model.

### Today, Part 1

1. What an array is
2. Making arrays
3. Stacking
4. Indexing, slicing — and views
5. Boolean masks, and finding *where*
6. dtypes, and `nan`
7. Vectorised math and `@`
8. Aggregation and `axis`
9. Transpose and reshape
10. Broadcasting
11. **NumPy on SBER 2025** — returns, volatility, drawdown, a portfolio, a simulation

> **Rule for this notebook.** Wherever you see **PREDICT**, write your answer down *before* running the cell. For NumPy the prediction is almost always a **shape**.

---
## 1 · What an array is

A NumPy array is a grid of numbers that all have the same type. That one restriction is what makes it fast, and it is why the rules differ from Python lists.

To use Numpy, we first need to import the `numpy` package. By convention, we import it using the alias `np`. Then, when we want to use modules or functions in this library, we preface them with `np.`

In [ ]:
import numpy as np

### Arrays and array construction

A numpy array is a grid of values, all of the same type, and is indexed by a tuple of nonnegative integers. The number of dimensions is the rank of the array; the shape of an array is a tuple of integers giving the size of the array along each dimension.

We can create a `numpy` array by passing a Python list to `np.array()`.

In [ ]:
a = np.array([1, 2, 3])  # Create a rank 1 array

This creates the array we can see on the right here:

![](http://jalammar.github.io/images/numpy/create-numpy-array-1.png)

In [ ]:
print(id(a), type(a), a.shape, a[0], a[1], a[2])

In [ ]:
a[0] = 5                 # Change an element of the array
print(id(a), a)

To create a `numpy` array with more dimensions, we can pass nested lists, like this:

![](http://jalammar.github.io/images/numpy/numpy-array-create-2d.png)

![](http://jalammar.github.io/images/numpy/numpy-3d-array.png)

In [ ]:
b = np.array(
    [
        [1, 2],
        [3, 4]
    ]
) # Create a rank 2 array
print(b)

In [ ]:
print(b.shape)

---
## 2 · Making arrays

`zeros`, `ones`, `full`, `eye`, a random generator, and two ways to make a sequence.

There are often cases when we want numpy to initialize the values of the array for us. numpy provides methods like `ones()`, `zeros()`, and `random.random()` for these cases. We just pass them the number of elements we want it to generate:

![](http://jalammar.github.io/images/numpy/create-numpy-array-ones-zeros-random.png)

We can also use these methods to produce multi-dimensional arrays, as long as we pass them a tuple describing the dimensions of the matrix we want to create:

![](http://jalammar.github.io/images/numpy/numpy-matrix-ones-zeros-random.png)

![](http://jalammar.github.io/images/numpy/numpy-3d-array-creation.png)

Sometimes, we need an array of a specific shape with “placeholder” values that we plan to fill in with the result of a computation. The `zeros` or `ones` functions are handy for this:

In [ ]:
a = np.zeros((2, 2))  # Create an array of all zeros
print(a)

In [ ]:
b = np.ones((1, 2)) # Create an array of all ones
print(b)

In [ ]:
c = np.full((2, 2), 7)  # Create a constant array
print(c)

In [ ]:
d = np.eye(3)  # 3x3 identity matrix: ones on the diagonal, zeros elsewhere
print(d)

**Random numbers.** The modern way is a *generator* with a seed. The same seed gives the same numbers on every run, on every laptop, which is what makes a result reproducible.

You will still see `np.random.random(...)` and `np.random.seed(...)` in older code and in model output. They work, but new code should use `default_rng`. Either way, **ask the model for a seed.**

In [ ]:
rng = np.random.default_rng(42)         # a seeded generator: same numbers on every run
print(rng.random((2, 2)))               # uniform on [0, 1), shape (2, 2)
print(rng.normal(0, 0.01, size=5))      # five "daily returns" with 1 % volatility

Numpy also has two useful functions for creating sequences of numbers: `arange` and `linspace`.

The `arange` function accepts three arguments, which define the start value, stop value of a half-open interval, and step size. (The default step size, if not explicitly specified, is 1; the default start value, if not explicitly specified, is 0.)

The `linspace` function is similar, but we can specify the number of values instead of the step size, and it will create a sequence of evenly spaced values.

In [ ]:
print(np.arange(5))           # start defaults to 0
print(np.arange(2, 10, 2))    # 10 is NOT included: the half-open rule from Lecture 2 again

**PREDICT.** How many numbers does `np.arange(0, 1, 0.25)` give, and is `1.0` among them? How many does `np.linspace(0, 1, 5)` give, and is `1.0` among them?

In [ ]:
print(np.arange(0, 1, 0.25))     # step given: the stop value is excluded
print(np.linspace(0, 1, 5))      # count given: both ends are included

`arange` takes a **step** and excludes the stop. `linspace` takes a **count** and includes both ends. With a fractional step, rounding can add or drop the last point in `arange`, so for evenly spaced floats prefer `linspace`.

---
## 3 · Stacking

Glue arrays together: `vstack` puts them on top of each other, `hstack` side by side. (`row_stack` is deprecated since NumPy 2.0 — use `vstack`.)

Sometimes, we may want to construct an array from existing arrays by “stacking” the existing arrays, either vertically or horizontally. We can use `vstack()` (or `row_stack`) and `hstack()` (or `column_stack`), respectively.

In [ ]:
a = np.array([1, 2, 3])
b = np.array([4, 5, 6])


In [ ]:
np.vstack((a, b))

In [ ]:
np.hstack((a, b))

In [ ]:
a = np.array([[7], [8], [9]])
b = np.array([[4], [5], [6]])
np.hstack((a,b))

---
## 4 · Indexing, slicing — and views

Slicing works as it does for lists, once per dimension: `a[rows, columns]`. **The important difference from lists is in cell 3 of this section.**

### Array indexing

Numpy offers several ways to index into arrays.

We can index and slice numpy arrays in all the ways we can slice Python lists:

![](http://jalammar.github.io/images/numpy/numpy-array-slice.png)

And you can index and slice numpy arrays in multiple dimensions. If slicing an array with more than one dimension, you should specify a slice for each dimension:

![](http://jalammar.github.io/images/numpy/numpy-matrix-indexing.png)

Create the following rank 2 array with shape (3, 4)
```python
[[ 1  2  3  4]
 [ 5  6  7  8]
 [ 9 10 11 12]]
```

Use slicing to pull out the subarray consisting of the first 2 rows and columns 1 and 2;

b is the following array of shape (2, 2):
```python
[[2 3]
 [6 7]]
```

In [ ]:
a = np.array(
    [
        [1, 2, 3, 4],
        [5, 6, 7, 8],
        [9, 10, 11, 12]
    ]
)
print(a)

b = a[:2, 1:3]
print(b)

A slice of an array is a view into the same data, so modifying it will modify the original array.

In [ ]:
print(a[0, 1])
b[0, 0] = 77    # b[0, 0] is the same piece of data as a[0, 1]
print(a[0, 1])

### A slice is a view: who owns the data?

In Lecture 2 a list slice `lst[2:]` made a **new list**. A NumPy slice does **not**: it is a second name for the same numbers in memory. This is the "who owns the list?" question again, with the opposite answer.

**PREDICT.** What does `prices` print after the cell below runs?

In [ ]:
prices = np.array([100.0, 101.0, 102.0, 103.0])
last_two = prices[2:]            # a view: no data is copied
safe     = prices[2:].copy()     # a real, independent copy

last_two[0] = 0.0
print("prices  :", prices)
print("safe    :", safe)

If you want to change a piece of an array without touching the original, say `.copy()`. When a model writes a function that modifies a slice of its input, the input changes too.

You can also mix integer indexing with slice indexing. However, doing so will yield an array of lower rank than the original array. Note that this is quite different from the way that MATLAB handles array slicing:

In [ ]:
a = np.array(
    [
        [1, 2, 3, 4],
        [5, 6, 7, 8],
        [9, 10, 11, 12]
    ]
)
print(a)

Two ways of accessing the data in the middle row of the array. Mixing integer indexing with slices yields an array of lower rank, while using only slices yields an array of the same rank as the original array:

In [ ]:
row_r1 = a[1, :]    # Rank 1 view of the second row of a
row_r2 = a[1:2, :]  # Rank 2 view of the second row of a
row_r3 = a[[1], :]  # Rank 2 view of the second row of a
print(row_r1, row_r1.shape)
print(row_r2, row_r2.shape)
print(row_r3, row_r3.shape)

In [ ]:
# We can make the same distinction when accessing columns of an array:
col_r1 = a[:, 1]
col_r2 = a[:, 1:2]
print(col_r1, col_r1.shape)
print(col_r2, col_r2.shape)

Integer array indexing: When you index into numpy arrays using slicing, the resulting array view will always be a subarray of the original array. In contrast, integer array indexing allows you to construct arbitrary arrays using the data from another array. Here is an example:

In [ ]:
a = np.array(
    [
        [1, 2],
        [3, 4],
        [5, 6],
    ]
)
a

In [ ]:
# An example of integer array indexing.
# The returned array will have shape (3,) and
print(a[[0, 1, 2], [0, 1, 0]])

# The above example of integer array indexing is equivalent to this:
print(np.array([a[0, 0], a[1, 1], a[2, 0]]))

In [ ]:
# When using integer array indexing, you can reuse the same
# element from the source array:
print(a[[0, 0], [1, 1]])

# Equivalent to the previous integer array indexing example
print(np.array([a[0, 1], a[0, 1]]))

**Rule of thumb.** A slice (`a[1:3]`, `a[:, 0]`) gives a **view**. Indexing with a list of positions (`a[[0, 2]]`) or with a mask (`a[a > 2]`, next section) gives a **copy**.

---
## 5 · Boolean masks, and finding *where*

A comparison on an array gives an array of `True`/`False`, called a **mask**. You can index with it, count it, and average it.

Boolean array indexing: Boolean array indexing lets you pick out arbitrary elements of an array. Frequently this type of indexing is used to select the elements of an array that satisfy some condition. Here is an example:

In [ ]:
a = np.array(
    [
        [1, 2],
        [3, 4],
        [5, 6],
    ]
)
print(a)
bool_idx = (a > 2)
print(bool_idx)

In [ ]:
# We use boolean array indexing to construct a rank 1 array
# consisting of the elements of a corresponding to the True values
# of bool_idx
# print(a[bool_idx])

# We can do all of the above in a single concise statement:
print(a[a > 2])

When working with numpy arrays, it’s often helpful to get the *indices* (not only the values) of array elements that meet certain conditions. There are a few numpy functions that you’ll definitely want to remember:

-   [`argmax`](https://numpy.org/doc/stable/reference/generated/numpy.argmax.html) (get index of maximum element in array)
-   [`argmin`](https://numpy.org/doc/stable/reference/generated/numpy.argmin.html) (get index of minimum element in array)
-   [`argsort`](https://numpy.org/doc/stable/reference/generated/numpy.argsort.html) (get sorted list of indices, by element value, in ascending order)
-   [`where`](https://numpy.org/doc/stable/reference/generated/numpy.where.html) (get indices of elements that meet some condition)

In [ ]:
a = np.array(
    [
        [1, 2, 3],
        [4, 5, 6,],
        [7, 8, 9],
        [10, 11, 12]
    ]
)

print(a)

In [ ]:
print(np.where(a > 5))

In [ ]:
a[np.where(a > 5)]

In [ ]:
a > 5

In [ ]:
a = np.array([1, 8, 9, -3, 2, 4, 7, 9])

# Get the index of the maximum element in a
print(np.argmax(a))

# Get the index of the minimum element in a
# (this array has two elements with the maximum value -
# only one index is returned)
print(np.argmin(a))

# Get sorted list of indices
print(np.argsort(a))

# Get sorted list of indices in descending order
# [::-1] is a special slicing index that returns the reversed list
print(np.argsort(a)[::-1])

# Get indices of elements that meet some condition
# this returns a tuple, the list of indices is the first entry
# so we use [0] to get it
print(np.where(a > 5)[0])

# Get indices of elements that meet some condition
# this example shows how to get the index of *all* the max values
print(np.where(a >= a[np.argmax(a)])[0])

For brevity we have left out a lot of details about numpy array indexing; if you want to know more you should read the documentation.

**The mean of a mask is a share.** `True` counts as 1 and `False` as 0, so `.sum()` counts and `.mean()` gives a proportion. This is how "share of up days" is written in NumPy.

In [ ]:
r = np.array([0.012, -0.004, 0.0, 0.007, -0.011])   # five daily returns
up = r > 0
print(up)
print("up days:", up.sum(), "of", up.size, "->", up.mean())

---
## 6 · dtypes, and `nan`

An array has **one** type for all its elements. NumPy picks it for you, and it does not warn you when that choice loses information.

### Datatypes

Every numpy array is a grid of elements of the same type. Numpy provides a large set of numeric datatypes that you can use to construct arrays. Numpy tries to guess a datatype when you create an array, but functions that construct arrays usually also include an optional argument to explicitly specify the datatype. Here is an example:

In [ ]:
x = np.array([1, 2])  # Let numpy choose the datatype
y = np.array([1.0, 2.0])  # Let numpy choose the datatype
z = np.array([1, 2], dtype=np.int32)  # Force a particular datatype

print(x.dtype, y.dtype, z.dtype)

**PREDICT.** Two cells. What is printed?

In [ ]:
shares = np.array([10, 20, 30])          # dtype chosen from the list: integers
shares[0] = 2.7                          # put a float into an integer array
print(shares, shares.dtype)

mixed = np.array([1, "2", 3.0])          # one string makes everything a string
print(mixed, mixed.dtype)

The `2.7` was silently cut to `2`. The mixed list became strings, so `mixed + 1` would fail. Both happen when a CSV is read with the wrong settings, so check `.dtype` right after loading.

**Missing values.** A missing number in a float array is `np.nan` ("not a number"). Any arithmetic with `nan` gives `nan`, and `nan` is not even equal to itself.

In [ ]:
r = np.array([0.01, np.nan, -0.02, 0.03])
print(np.mean(r))        # nan: one missing day poisons the average
print(np.nanmean(r))     # skips the nan
print(np.nan == np.nan)  # False, so use np.isnan to find them
print(np.isnan(r))

---
## 7 · Vectorised math and `@`

Arithmetic on arrays works element by element, with no loop. `*` multiplies element by element; `@` is the matrix product.

### Array math

What makes working with `numpy` so powerful and convenient is that it comes with many *vectorized* math functions for computation over elements of an array. These functions are highly optimized and are *very* fast - much, much faster than using an explicit `for` loop.

For example, let’s create a large array of random values and then sum it both ways. We’ll use a `%%time` *cell magic* to time them.

In [ ]:
rng = np.random.default_rng(0)
a = rng.random(10_000_000)       # ten million numbers
a.shape

In [ ]:
%%time
x = np.sum(a)

In [ ]:
%%time
x = 0
for element in a:
    x = x + element

Look at the “Wall Time” in the output - note how much faster the vectorized version of the operation is! This type of fast computation is a major enabler of machine learning, which requires a *lot* of computation.

Whenever possible, we will try to use these vectorized operations.

Some mathematic functions are available both as operator overloads and as functions in the numpy module.

For example, you can perform an elementwise sum on two arrays using either the + operator or the `add()` function.

![](http://jalammar.github.io/images/numpy/numpy-arrays-adding-1.png)

![](http://jalammar.github.io/images/numpy/numpy-matrix-arithmetic.png)

In [ ]:
a = [1, 2]
b = [3, 4]
c = [0, 0]
for i in range(2):
  c[i] = a[i] + b[i]

c

**PREDICT.** `a` and `b` are Python **lists** here. What does `a + b` give?

In [ ]:
a + b

In [ ]:
x = np.array([[1, 2], [3, 4]], dtype=np.float64)
y = np.array([[5, 6], [7, 8]], dtype=np.float64)

# Elementwise sum; both produce the array
print(x + y)
print(np.add(x, y))

And this works for other operations as well, not only addition:

![](http://jalammar.github.io/images/numpy/numpy-array-subtract-multiply-divide.png)

In [ ]:
# Elementwise difference; both produce the array
print(x - y)
print(np.subtract(x, y))

In [ ]:
# Elementwise product; both produce the array
print(x * y)
print(np.multiply(x, y))

In [ ]:
# Elementwise division; both produce the array
# [[ 0.2         0.33333333]
#  [ 0.42857143  0.5       ]]
print(x / y)
print(np.divide(x, y))

In [ ]:
# Elementwise square root; produces the array
# [[ 1.          1.41421356]
#  [ 1.73205081  2.        ]]
print(np.sqrt(x))

Note that unlike MATLAB, `*` is elementwise multiplication, not matrix multiplication. We instead use the `dot()` function to compute inner products of vectors, to multiply a vector by a matrix, and to multiply matrices. `dot()` is available both as a function in the numpy module and as an instance method of array objects:

![](http://jalammar.github.io/images/numpy/numpy-matrix-dot-product-1.png)

In [ ]:
x = np.array([[1, 2], [3, 4]])
y = np.array([[5, 6], [7, 8]])

v = np.array([9, 10])
w = np.array([11, 12])

# Inner product of vectors; both produce 219
print(v.dot(w))
print(np.dot(v, w))

You can also use the `@` operator which is equivalent to numpy's `matmul` (and is similar to numpy's `dot`) operator.

In [ ]:
v * w

In [ ]:
print(v @ w)

In [ ]:
# Matrix / vector product; both produce the rank 1 array [29 67]
print(x.dot(v))
print(np.dot(x, v))
print(x @ v)

In [ ]:
# Matrix / matrix product; both produce the rank 2 array
# [[19 22]
#  [43 50]]
print(x.dot(y))
print(np.dot(x, y))
print(x @ y)

You may meet `np.matrix` in old code or in model output, where `*` means the matrix product. Don't use it: the NumPy documentation discourages it, and mixing it with ordinary arrays changes what `*` means. Use ordinary arrays and `@`.

---
## 8 · Aggregation and `axis`


`sum`, `mean`, `max`, `std` of everything, or along one axis. **`axis=0` collapses the rows: you get one number per column.** `axis=1` collapses the columns: one number per row.

Besides for the functions that overload operators, Numpy also provides many useful functions for performing computations on arrays, such as `min()`, `max()`, `sum()`, and others:

![](http://jalammar.github.io/images/numpy/numpy-matrix-aggregation-1.png)

In [ ]:
x = np.array([[1, 2], [3, 4], [5, 6]])

print(np.max(x))  # Compute max of all elements; prints "6"
print(np.min(x))  # Compute min of all elements; prints "1"
print(np.sum(x))  # Compute sum of all elements; prints "21"

Not only can we aggregate all the values in a matrix using these functions, but we can also aggregate across the rows or columns by using the `axis` parameter:

![](http://jalammar.github.io/images/numpy/numpy-matrix-aggregation-4.png)

In [ ]:
x

In [ ]:
np.sum(x, axis=1)

In [ ]:
print(np.max(x, axis=0))  # max of each column: [5 6]
print(np.max(x, axis=1))  # max of each row:    [2 4 6]

You can find the full list of mathematical functions provided by numpy in the [documentation](http://docs.scipy.org/doc/numpy/reference/routines.math.html).

**PREDICT.** In section 11 we load a price table with one row per trading day and four columns (open, high, low, close), so its shape is `(254, 4)`. What are the shapes of `m.mean(axis=0)` and `m.mean(axis=1)`, and which one is "the average close"?

In [ ]:
m = np.ones((254, 4))
print(m.mean(axis=0).shape)
print(m.mean(axis=1).shape)
print(m.mean().shape)            # no axis: one number for the whole table

---
## 9 · Transpose and reshape

`.T` swaps rows and columns. `.reshape` changes the shape without changing the numbers. `-1` means "work this one out for me".

Apart from computing mathematical functions using arrays, we frequently need to reshape or otherwise manipulate data in arrays. The simplest example of this type of operation is transposing a matrix; to transpose a matrix, simply use the T attribute of an array object.

![](http://jalammar.github.io/images/numpy/numpy-transpose.png)

In [ ]:
x = np.array([[1, 2], [3, 4], [5, 6]])

print(x)
print("transpose\n", x.T)

In [ ]:
v = np.array([[1,2,3]])
print(v)
print("transpose\n", v.T)

In more advanced use case, you may find yourself needing to change the dimensions of a certain matrix. This is often the case in machine learning applications where a certain model expects a certain shape for the inputs that is different from your dataset. numpy's `reshape()` method is useful in these cases.

![](http://jalammar.github.io/images/numpy/numpy-reshape.png)

A common task in this class will be to convert a 1D array to a 2D array, and vice versa. We can use `reshape()` for this.

For example, suppose we had this 2D array, but we need to pass it to a function that expects a 1D array.

In [ ]:
w = np.array([[1], [2], [3]])
print(w)
w.shape

We can remove the “unnecessary” extra dimension with

In [ ]:
y = w.reshape(-1,)
print(y)
y.shape

Note that we can pass -1 as one dimension and numpy will infer the correct size based on our matrix size!

There’s also a `squeeze()` function that removes *all* of the “unnecessary” dimensions (dimensions that have size 1) from an array:

In [ ]:
z = w.squeeze()
print(z)
z.shape

To go from a 1D to 2D array, we can just add in another dimension of size 1:

In [ ]:
y.reshape((-1, 1))

---
## 10 · Broadcasting

How NumPy combines arrays of **different** shapes. Three rules, read right to left through the shapes.

### Broadcasting

Broadcasting is a powerful mechanism that allows numpy to work with arrays of different shapes when performing arithmetic operations.

For example: basic linear algebra, we can only add (and perform similar element-wise operations) two matrics that have the *same* dimension. In numpy, if we want to add two matrics that have different dimensions, numpy will implicitly “extend” the dimension of one matrix to match the other so that we can perform the operation.

So these operations will work, instead of returning an error:

![](https://sebastianraschka.com/images/blog/2020/numpy-intro/broadcasting-1.png)

![](https://sebastianraschka.com/images/blog/2020/numpy-intro/broadcasting-2.png)

> ⚠️ **This cell is supposed to raise an error.** That is the lesson. Run it, read the error type and the message, and move on — nothing is broken.

In [ ]:
a = [1, 2, 3]
a + 1

In [ ]:
a = np.array([1, 2, 3])
a + 1

Broadcasting two arrays together follows these rules:

**Rule 1**: If the two arrays differ in their number of dimensions, the shape of the one with fewer dimensions is padded with ones on its leading (left) side.

For example, in the following cell, `a` will be implicitly extended to shape (1,3):

In [ ]:
a = np.array([1, 2, 3])         # has shape (3,): one dimension
b = np.array([[4], [5], [6]]) # has shape (3,1): two dimensions
c = a + b                     # will have shape (3,3) (two dimensions)
print(c)

**Rule 2**: If the shape of the two arrays does not match in any dimension, the array with shape equal to 1 in that dimension is stretched to match the other shape.

For example, in the following cell `a` will be implicitly extended to shape (3,2):

In [ ]:
a = np.array([[1],[2],[3]])         # has shape (3,1)
b = np.array([[4,5], [6,7], [8,9]]) # has shape (3,2)
c = a + b                           # will have shape (3,2)
print(c)

**Rule 3**: If in any dimension the sizes disagree and neither is equal to 1, an error is raised:

> ⚠️ **This cell is supposed to raise an error.** That is the lesson. Run it, read the error type and the message, and move on — nothing is broken.

In [ ]:
a = np.array([[1], [2], [3], [4]])      # has shape (4,1)
b = np.array([[4, 5], [6, 7], [8, 9]])  # has shape (3,2)
c = a + b                            # ValueError: operands could not be broadcast

For more detail, you can read the explanation from the [documentation](http://docs.scipy.org/doc/numpy/user/basics.broadcasting.html).

Functions that support broadcasting are known as universal functions. You can find the list of all universal functions in the [documentation](http://docs.scipy.org/doc/numpy/reference/ufuncs.html#available-ufuncs).

Here are a few visual examples involving broadcasting.

![](http://jalammar.github.io/images/numpy/numpy-array-broadcast.png)

Note that these arrays are compatible in each dimension if they have either the same size in that dimension, or if one array has size 1 in that dimension.

![](http://jalammar.github.io/images/numpy/numpy-matrix-broadcast.png)

And here are some more practical applications:

In [ ]:
# Compute outer product of vectors

v = np.array([1, 2, 3])  # v has shape (3,)
w = np.array([4, 5])    # w has shape (2,)

# To compute an outer product, we first reshape v to be a column
# vector of shape (3, 1); we can then broadcast it against w to yield
# an output of shape (3, 2), which is the outer product of v and w:

print(np.reshape(v, (3, 1)) * w)

In [ ]:
# Add a vector to each row of a matrix
x = np.array([[1, 2, 3], [4, 5, 6]])
# x has shape (2, 3) and v has shape (3,) so they broadcast to (2, 3),
# giving the following matrix:

print(x + v)

In [ ]:
# Add a vector to each column of a matrix
# x has shape (2, 3) and w has shape (2,).
# If we transpose x then it has shape (3, 2) and can be broadcast
# against w to yield a result of shape (3, 2); transposing this result
# yields the final result of shape (2, 3) which is the matrix x with
# the vector w added to each column. Gives the following matrix:

print((x.T + w).T)

In [ ]:
# Another solution is to reshape w to be a column vector of shape (2, 1);
# we can then broadcast it directly against x to produce the same output.
print(x + np.reshape(w, (2, 1)))

In [ ]:
# Multiply a matrix by a constant:
# x has shape (2, 3). Numpy treats scalars as arrays of shape ();
# these can be broadcast together to shape (2, 3), producing the
# following array:
print(x * 2)

Broadcasting typically makes your code more concise and faster, so you should strive to use it where possible.

### The case you will meet most: subtract an average

Demeaning (subtracting the average) is everywhere in finance: excess returns, z-scores, covariance. Take a table `X` of shape `(5, 3)`: five days, three assets.

**PREDICT.** `X.mean(axis=0)` has shape `(3,)` and `X.mean(axis=1)` has shape `(5,)`. Using the three rules, which of the two can be subtracted from `X` directly?

In [ ]:
X = np.arange(15.0).reshape(5, 3)
col_means = X.mean(axis=0)                       # (3,)
row_means = X.mean(axis=1)                       # (5,)
print(X.shape, col_means.shape, row_means.shape)

print(X - col_means)                             # (5, 3) - (3,): rule 1, then rule 2
print(X - X.mean(axis=1, keepdims=True))         # keepdims=True keeps shape (5, 1)

`(5, 3) - (3,)` works: each **column** loses its own mean. `(5, 3) - (5,)` does not, because the shapes are aligned from the right, so 3 meets 5. The fix is `keepdims=True`, which keeps the result as a column `(5, 1)`.

---
## 11 · NumPy on SBER 2025

The same SBER file as in Lectures 1–3: 254 trading days of 2025. In Lecture 3 each of these took a loop. Here each is one or two lines, and after each one you check the shape and assert something you already know.

In [ ]:
# Data loader. Tries the local cached file first, then the course repository.
import os, urllib.request
import numpy as np

DATA_URL = "https://raw.githubusercontent.com/kosavina/hse-python-2026/main/data/SBER_2025_daily.csv"
DATA_FILE = "SBER_2025_daily.csv"
for candidate in (DATA_FILE, os.path.join("data", DATA_FILE), os.path.join("..", "data", DATA_FILE)):
    if os.path.exists(candidate):
        DATA_FILE = candidate
        break
else:
    urllib.request.urlretrieve(DATA_URL, DATA_FILE)

# Two arrays: the dates as strings, and the four price columns as floats.
# Columns in the file: TRADEDATE, OPEN, HIGH, LOW, CLOSE, VOLUME, VALUE
dates = np.loadtxt(DATA_FILE, delimiter=",", skiprows=1, usecols=0, dtype=str)
ohlc  = np.loadtxt(DATA_FILE, delimiter=",", skiprows=1, usecols=(1, 2, 3, 4))

print("dates:", dates.shape, dates.dtype)
print("ohlc :", ohlc.shape, ohlc.dtype)
print(dates[0],  ohlc[0])
print(dates[-1], ohlc[-1])

assert dates.shape == (254,) and ohlc.shape == (254, 4)
assert (ohlc[:, 1] >= ohlc[:, 2]).all(), "a HIGH below its LOW would mean the columns are mixed up"

### Daily returns

`close[1:]` is "today" and `close[:-1]` is "yesterday", two slices of the same array shifted by one day. Dividing them lines each day up with the day before.

**PREDICT.** What is `r.shape`?

In [ ]:
close = ohlc[:, 3]                  # column 3 is CLOSE
r = close[1:] / close[:-1] - 1      # simple daily returns
print(close.shape, r.shape)
print(r[:3])

r_loop = []
for i in range(1, len(close)):
    r_loop.append(close[i] / close[i - 1] - 1)
assert np.allclose(r, r_loop)

254 prices give **253** returns, because the first day has no "yesterday". Keep that in mind whenever a return series has to be lined up with the dates.

### Total return, two ways

Chaining the daily returns with `np.prod(1 + r)` has to reproduce the first-to-last price change. That is a free check, so use it.

In [ ]:
total_direct = close[-1] / close[0] - 1
total_chain  = np.prod(1 + r) - 1
print(f"{total_direct:.2%}   {total_chain:.2%}")
assert np.isclose(total_direct, total_chain)

**10.16%**: the same figure as in Lectures 1 and 3.

### Up days, best day, worst day

**PREDICT.** Why does the code use `dates[1:]` and not `dates` to look up the date of the best day?

In [ ]:
print("up days:", (r > 0).sum(), "of", r.size, f"= {(r > 0).mean():.1%}")

ret_dates = dates[1:]              # return i ends on day i + 1
i_best, i_worst = r.argmax(), r.argmin()
print("best :", ret_dates[i_best],  f"{r[i_best]:+.2%}")
print("worst:", ret_dates[i_worst], f"{r[i_worst]:+.2%}")

Expected: 129 up days of 253 (51.0%), best day 2025-02-12 (+6.78%), worst day 2025-04-04 (-5.54%). With `dates` instead of `dates[1:]` every date would be one trading day early, and nothing would raise an error.

### Volatility, and the `ddof` detail

The annualised volatility is the standard deviation of the daily returns times √252 (the usual count of trading days in a year).

`np.std` divides by *n* by default (`ddof=0`). pandas and most textbooks divide by *n − 1* (`ddof=1`). When the model mixes NumPy and pandas, this is where two "identical" numbers stop agreeing.

In [ ]:
for ddof in (0, 1):
    daily = r.std(ddof=ddof)
    print(f"ddof={ddof}:  daily {daily:.4%}   annualised {daily * np.sqrt(252):.2%}")

Expected: 22.44% with `ddof=0` and 22.48% with `ddof=1`. The gap is small with 253 observations and larger with 20, so **say which one you want** in the prompt.

### Maximum drawdown

The worst fall from a previous peak. `np.maximum.accumulate` gives the running maximum, the highest close so far on each day.

In [ ]:
peak = np.maximum.accumulate(close)           # same shape as close
drawdown = close / peak - 1                   # 0 on a new high, negative below it

i_trough = drawdown.argmin()
i_peak = close[:i_trough + 1].argmax()        # the peak it fell from
print(f"max drawdown {drawdown[i_trough]:.2%}")
print("from", dates[i_peak], close[i_peak], "to", dates[i_trough], close[i_trough])

assert peak.shape == close.shape
assert (drawdown <= 0).all() and drawdown[0] == 0

Expected: -13.81%, from 2025-07-17 (327.05) to 2025-10-08 (281.9).

### `axis` and broadcasting on the whole table

**PREDICT.** Shapes of `ohlc.max(axis=0)`, `day_range`, and `ohlc / ohlc[0]`?

In [ ]:
print("year high per column:", ohlc.max(axis=0))    # OPEN, HIGH, LOW, CLOSE
print("year low per column :", ohlc.min(axis=0))

day_range = (ohlc[:, 1] - ohlc[:, 2]) / ohlc[:, 3]  # (HIGH - LOW) / CLOSE, one per day
print(day_range.shape, f"average daily range {day_range.mean():.2%}")

normalised = ohlc / ohlc[0]         # (254, 4) / (4,): every column starts at 1.0
print(normalised.shape, normalised[-1])
assert np.isclose(normalised[-1, 3] - 1, total_direct)

### A portfolio: `@` and a closed-form check

Three assets with stated volatilities and correlations (illustrative numbers, not SBER). The covariance matrix is `Σᵢⱼ = σᵢ ρᵢⱼ σⱼ`, and the portfolio volatility is `√(w Σ w)`.

The covariance line is broadcasting: `(3, 1) * (3, 3) * (3,)`. After it, the result is checked two independent ways.

In [ ]:
vols = np.array([0.30, 0.20, 0.10])            # annual volatility of each asset
corr = np.array([[1.0, 0.6, 0.2],
                 [0.6, 1.0, 0.3],
                 [0.2, 0.3, 1.0]])
cov = vols[:, None] * corr * vols              # sigma_i * rho_ij * sigma_j
assert np.allclose(cov, np.diag(vols) @ corr @ np.diag(vols))

w = np.array([0.5, 0.3, 0.2])                  # weights, sum to 1
port_vol = np.sqrt(w @ cov @ w)
print(f"portfolio volatility {port_vol:.2%}")

# Closed-form check with two assets: sigma^2 = w1^2 s1^2 + w2^2 s2^2 + 2 w1 w2 rho s1 s2
w2 = np.array([0.6, 0.4, 0.0])
closed_form = np.sqrt(0.6**2 * 0.30**2 + 0.4**2 * 0.20**2 + 2 * 0.6 * 0.4 * 0.6 * 0.30 * 0.20)
assert np.isclose(np.sqrt(w2 @ cov @ w2), closed_form)

Expected: 19.80%. The weighted average of the three volatilities, `w @ vols`, is 23.00%. The gap between the two is diversification: the assets are not perfectly correlated.

### A simulation: 10 000 possible years

A **bootstrap**: build each future year by drawing 252 of SBER's own 2025 daily returns at random, with replacement. This is a toy, because it assumes every day is independent of the day before. It is still a good exercise in shapes and `axis`.

**PREDICT.** The shapes of `sampled`, `paths` and `final`. And what would `axis=0` in `cumprod` compute instead?

In [ ]:
rng = np.random.default_rng(2026)
n_paths, n_days = 10_000, 252

sampled = rng.choice(r, size=(n_paths, n_days))       # one row per simulated year
paths = close[-1] * np.cumprod(1 + sampled, axis=1)   # compound along each row
final = paths[:, -1]                                  # the last day of every path

print(sampled.shape, paths.shape, final.shape)
print("5 % / 50 % / 95 % of the final price:", np.percentile(final, [5, 50, 95]).round(1))
print(f"share of paths that end below today's close: {(final < close[-1]).mean():.1%}")

# Sanity check: the median log growth should be close to 252 x the average daily log return.
assert abs(np.median(np.log(final / close[-1])) - np.log(1 + r).mean() * 252) < 0.03

With `axis=0` NumPy would compound **down the columns**, across the 10 000 simulated years instead of through the 252 days of each. The shapes would be the same, nothing would raise, and every number would be wrong. That is why "which axis?" is the first question to ask about any NumPy line the model writes.

---
## Wrap-up: what to recognise, and how to ask

| You want | NumPy |
|---|---|
| make an array | `np.array`, `np.zeros`, `np.ones`, `np.full`, `np.eye`, `np.arange`, `np.linspace` |
| random, reproducibly | `rng = np.random.default_rng(seed)`, `rng.random`, `rng.normal`, `rng.choice` |
| inspect | `.shape`, `.dtype`, `.ndim`, `.size` |
| pick | `a[i]`, `a[i:j]`, `a[:, k]`, `a[mask]`, `a.copy()` |
| find | `argmax`, `argmin`, `argsort`, `np.where` |
| combine | `np.vstack`, `np.hstack` |
| compute | `+ - * /` elementwise, `@` matrix product, `np.sqrt`, `np.log`, `np.exp` |
| aggregate | `sum`, `mean`, `std(ddof=…)`, `max`, `min`, `prod`, `cumprod`, `np.percentile` — with `axis=` |
| missing values | `np.nan`, `np.isnan`, `np.nanmean` |
| running maximum | `np.maximum.accumulate` |
| reshape | `.T`, `.reshape`, `-1`, `keepdims=True`, `[:, None]` |

**When you ask a model for NumPy code:**

1. **Give the input shapes**, e.g. "`ohlc` is a `(254, 4)` float array with columns open, high, low, close".
2. **Ask it to print the shape** of every intermediate array.
3. **Name the axis** in words: "for each day" means across the columns, and "for each asset" means down the rows.
4. **Say `ddof`, the seed and the annualisation factor**, or you will get whatever the model assumed.
5. **Add one `assert` you know is true**: a known total, a closed form, the loop version.

That is the toolkit for Part 2.